# 6장 실습 ② — 완화 기법 다섯 가지

**TensorFlow 판**

드롭아웃·L2 규제·배치 정규화·조기 종료를 하나씩 켜 가며 비교합니다.

> **모든 규제 기법은 학습 정확도를 떨어뜨립니다.** 그것이 목적입니다.
> 걸었는데 학습 정확도가 그대로면 안 걸린 것입니다.

## 6.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 6.1 실험대 — 잡음을 키운 소용돌이

5장에서는 잡음 0.06을 썼습니다. 여기서는 **0.25**로 키웁니다.
잡음이 있어야 *"잡음까지 외우는"* 현상이 보이기 때문입니다.

In [ ]:
def make(n, seed=42, noise=0.25):
    """잡음을 0.25로 키운다 — 과적합이 눈에 보이도록."""
    x, y = data.spirals(n, seed=seed, noise=noise)
    return data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=seed)

s = make(1600)
print(s.summary())
plot.scatter2d(s.x_train, s.y_train, class_names=("무리 0", "무리 1"),
               xlabel="$x_1$", ylabel="$x_2$", title="잡음 0.25의 소용돌이")
plt.show()

## 6.2 학습 함수 — 여기만 판마다 다릅니다

드롭아웃·L2·배치 정규화·조기 종료를 옵션으로 받습니다.
**이 아래의 모든 셀은 세 판이 같습니다.**

In [ ]:
import tensorflow as tf

dlbook.set_seed(42)

def train(split, units=(256, 256, 256), drop=0.0, l2=0.0, bn=False,
          act="relu", lr=0.001, epochs=200, bs=32, early=0, seed=42):
    """모델을 만들어 학습시키고 (학습 정확도, 시험 정확도, history)를 돌려준다.

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    reg = tf.keras.regularizers.l2(l2) if l2 else None
    ls = [tf.keras.layers.Input(shape=(2,))]
    for u in units:
        ls.append(tf.keras.layers.Dense(u, activation=None if bn else act,
                                        kernel_regularizer=reg))
        if bn:
            ls += [tf.keras.layers.BatchNormalization(),
                   tf.keras.layers.Activation(act)]
        if drop:
            ls.append(tf.keras.layers.Dropout(drop))
    ls.append(tf.keras.layers.Dense(1, activation="sigmoid"))
    model = tf.keras.Sequential(ls)
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    cbs = ([tf.keras.callbacks.EarlyStopping(patience=early,
                                             restore_best_weights=True)]
           if early else [])
    h = model.fit(split.x_train, split.y_train,
                  validation_data=(split.x_val, split.y_val),
                  epochs=dlbook.smoke.epochs(epochs), batch_size=bs,
                  verbose=0, callbacks=cbs)
    def acc(xa, ya):
        p = (model.predict(xa, verbose=0).reshape(-1) > 0.5).astype("int64")
        return metrics.accuracy(ya, p)
    return acc(split.x_train, split.y_train), acc(split.x_test, split.y_test), h.history

## 6.3 완화 기법을 하나씩 켠다

In [ ]:
# 학습 데이터를 일부러 적게 두고, 완화 기법을 하나씩 켠다.
s_small = make(150)
print(f"학습 데이터 {len(s_small.x_train)}개, 은닉층 3개(각 256노드), 200 epoch")
print()
print(f"{'방법':<20}{'학습':>10}{'시험':>10}{'차이':>10}")

cases = [("아무것도 안 함", {}, "none"),
         ("Dropout 0.3", {"drop": 0.3}, "drop03"),
         ("Dropout 0.5", {"drop": 0.5}, "drop05"),
         ("L2 규제 0.01", {"l2": 0.01}, "l2"),
         ("배치 정규화", {"bn": True}, "bn"),
         ("Dropout + L2", {"drop": 0.3, "l2": 0.001}, "droptl2"),
         ("조기 종료(patience 20)", {"early": 20}, "early")]

for name, kw, key in cases:
    tr, te, h = train(s_small, **kw)
    note = f"   ({len(h['loss'])} epoch에서 멈춤)" if kw.get("early") else ""
    print(f"{name:<20}{tr:>10.3f}{te:>10.3f}{tr - te:>10.3f}{note}")
    dlbook.record(f"ch06_reg_{key}_test_acc", te)

print()
print("→ 모든 기법이 학습 정확도를 떨어뜨리고 시험 정확도를 올립니다. 그것이 목적입니다.")
print("→ 조기 종료만 예외입니다. patience가 짧으면 과소적합이 납니다.")

## 정리

- **모든 규제 기법은 학습 정확도를 떨어뜨리고 시험 정확도를 올립니다.**
- **조기 종료는 `patience` 를 짧게 주면 해가 됩니다.** 과적합을 막으려다
  과소적합을 만듭니다.
- **여러 개를 한꺼번에 켜지 마십시오.** 하나씩 켜고 검증 성능을 확인하십시오.

### 연습

1. 드롭아웃 비율을 0.0 ~ 0.9로 훑어 가장 좋은 값을 찾으십시오.
   0.9에서는 왜 나빠집니까.
2. 조기 종료의 `patience` 를 20, 50, 100으로 바꿔 보십시오.
3. PyTorch 판에서 예측 전에 `model.eval()` 을 **빼고** 두 번 예측해 보십시오.
   결과가 같습니까.